# Preference data pipeline — demo

End-to-end exercise of the Phase-C preference pipeline:

1. Generate 15 single-shot pairs from `Billiards4BallEnv` (seeds 100..114).
2. Generate 15 multi-shot pairs from `Billiards4BallInningEnv` with one step each (seeds 200..214).
3. Render the first three pairs as side-by-side dual HTML viewers, written to `artifacts/preference_demo/pair_*.html`.
4. Apply the heuristic labeler to all 30 pairs and print the preference distribution.
5. If `ANTHROPIC_API_KEY` is present, label the first 5 with Claude and report agreement with the heuristic.
6. Append the labeled pairs to `data/preference_pairs.jsonl`.
7. Print a final summary table.

Headless `nbconvert` runs of this notebook must succeed even without `ANTHROPIC_API_KEY` — the Claude step is gated.

In [ ]:
import os
from pathlib import Path

import numpy as np
from IPython.display import HTML, display

from billiards.env import Billiards4BallEnv
from billiards.inning_env import Billiards4BallInningEnv
from billiards.preference import (
    PreferencePair,
    append_pair,
    generate_random_pairs,
    load_pairs,
    summary_stats,
)
from billiards.preference.labeler_heuristic import (
    heuristic_score,
    label_pair_heuristic,
)
from billiards.render.dual_replay import render_dual_html

# Resolve the project root so file paths work whether the kernel is launched from
# `notebooks/` or from the repo root.
PROJECT_ROOT = Path('.').resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts' / 'preference_demo'
DATA_DIR = PROJECT_ROOT / 'data'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

have_api_key = bool(os.environ.get('ANTHROPIC_API_KEY'))
print('ANTHROPIC_API_KEY:', 'available' if have_api_key else 'skipped (Claude step will be a no-op)')

## 1. Single-shot pairs from `Billiards4BallEnv`

In [ ]:
single_pairs, single_trajs = generate_random_pairs(
    env_factory_or_env=lambda: Billiards4BallEnv(),
    n_pairs=15,
    seed=100,
)
for p in single_pairs:
    p.label_meta = {**p.label_meta, 'kind': 'single_shot'}
print(f'generated {len(single_pairs)} single-shot pairs')
print('first pair_id:', single_pairs[0].pair_id)
print('first pair result_A keys:', sorted(single_pairs[0].result_A.keys()))

## 2. Multi-shot pairs from `Billiards4BallInningEnv` (1 shot each)

In [ ]:
inning_pairs, inning_trajs = generate_random_pairs(
    env_factory_or_env=lambda: Billiards4BallInningEnv(max_shots=1),
    n_pairs=15,
    seed=200,
)
for p in inning_pairs:
    p.label_meta = {**p.label_meta, 'kind': 'inning_first_shot'}
print(f'generated {len(inning_pairs)} inning first-shot pairs')
all_pairs = single_pairs + inning_pairs
all_trajs = single_trajs + inning_trajs
print(f'total pairs: {len(all_pairs)}')

## 3. Render the first three pairs as dual HTML viewers

In [ ]:
spec_payload = {'width_m': 2.540, 'height_m': 1.270, 'ball_radius_m': 0.03275}
rendered_paths = []
for k in range(3):
    pair = all_pairs[k]
    tA, tB = all_trajs[k]
    desc_A = {
        'score': pair.result_A['score'],
        'fouled': pair.result_A['fouled'],
        'cushion_hits': pair.result_A['cushion_hits'],
        'duration': f"{pair.result_A['duration']:.2f}s",
        'events': pair.result_A.get('event_types', []),
    }
    desc_B = {
        'score': pair.result_B['score'],
        'fouled': pair.result_B['fouled'],
        'cushion_hits': pair.result_B['cushion_hits'],
        'duration': f"{pair.result_B['duration']:.2f}s",
        'events': pair.result_B.get('event_types', []),
    }
    out_path = ARTIFACTS_DIR / f'pair_{k:02d}_{pair.pair_id[:8]}.html'
    render_dual_html(
        pair_id=pair.pair_id,
        traj_A=tA, traj_B=tB,
        spec=spec_payload,
        descriptors_A=desc_A,
        descriptors_B=desc_B,
        save_path=out_path,
    )
    rendered_paths.append(out_path)
    print(f'rendered {out_path.relative_to(PROJECT_ROOT)} ({out_path.stat().st_size} bytes)')
rendered_paths

## 4. Heuristic labels (deterministic)

In [ ]:
from collections import Counter

heuristic_pairs = [label_pair_heuristic(p) for p in all_pairs]
dist = Counter(p.preference for p in heuristic_pairs)
print('heuristic preference distribution:')
for k in ('A', 'B', 'tie'):
    print(f"  {k:<4} {dist.get(k, 0)}")
scores = [(p.pair_id[:8], p.label_meta['score_A'], p.label_meta['score_B'], p.preference) for p in heuristic_pairs[:5]]
print('first 5 (pair_id, score_A, score_B, pref):')
for row in scores:
    print(' ', row)

## 5. Optional: Claude labels for the first 5 pairs

In [ ]:
claude_pairs = []
if have_api_key:
    from billiards.preference.labeler_ai import label_pair_with_claude
    for p in all_pairs[:5]:
        try:
            claude_pairs.append(label_pair_with_claude(p))
        except Exception as exc:
            print(f'  claude labeling failed for {p.pair_id[:8]}: {type(exc).__name__}: {exc}')
    h_first5 = [hp for hp in heuristic_pairs[:5]]
    matches = sum(1 for cp, hp in zip(claude_pairs, h_first5) if cp.preference == hp.preference)
    n = len(claude_pairs)
    print(f'claude labeled {n}/5 pairs')
    if n:
        print(f'agreement with heuristic: {matches}/{n} = {matches / n:.0%}')
    for cp in claude_pairs:
        print(f"  {cp.pair_id[:8]}: claude={cp.preference}")
else:
    print('skipping Claude labeling (ANTHROPIC_API_KEY not set)')

## 6. Persist to `data/preference_pairs.jsonl`

In [ ]:
jsonl_path = DATA_DIR / 'preference_pairs.jsonl'
if jsonl_path.exists():
    jsonl_path.unlink()  # fresh write per demo run
for p in heuristic_pairs:
    append_pair(p, path=jsonl_path)
for p in claude_pairs:
    append_pair(p, path=jsonl_path)
print(f'wrote {jsonl_path.relative_to(PROJECT_ROOT)} ({jsonl_path.stat().st_size} bytes)')
loaded = load_pairs(jsonl_path)
print(f'reloaded {len(loaded)} pairs from disk')

## 7. Summary

In [ ]:
stats = summary_stats(loaded)
print('summary_stats:')
for k, v in stats.items():
    print(f'  {k}: {v}')